In [2]:

from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer('all-MiniLM-L6-v2')

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='02-vector-search/faq_vectors2.db'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
query = "I just discorey the curse. Can I still join it?"
query_vector = model.encode(query)
results = vs_index.search(query_vector ,num_results=5)

In [7]:
results

[]

In [ ]:
from rag_helper import RAGBase

class RAGVector(RAGBase):
    def __init__(self, embedder,**kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query: str, num_results: int = 5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}
        results = self.index.search(
            query_vector, filter_dict=filter_dict,
            num_results=num_results)
        
        return results

In [22]:
from dotenv import load_dotenv
from openai import OpenAI
import os
load_dotenv()
# openai_client = OpenAI()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)


In [23]:
vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,

)

In [24]:
query2 = "The program has already begun, can I still sign up?"
vector_assistant.rag(query2)

"I don't know. There's not enough information provided in the context to determine if sign-ups are still possible."